## ElastiCache — Redis vs. Memcached

**ElastiCache** runs two engines, and choosing wrong is a common mistake. **Redis** (and the Valkey fork) offers rich **data structures** (strings, hashes, lists, sets, **sorted sets**, bitmaps, HyperLogLog, streams, geo), **persistence** (RDB snapshots + AOF), **replication with Multi-AZ automatic failover**, **cluster-mode sharding** (up to 500 shards), plus pub/sub, Lua, transactions, and backup/restore — mostly **single-threaded**. **Memcached** is deliberately simpler: **strings only**, **no persistence, no replication, no failover**, but **multi-threaded** (scales with CPU cores) and horizontally scaled by adding nodes (client-side consistent hashing + Auto Discovery). **The default is Redis**; pick Memcached only for genuinely simple key-value caching where multi-thread throughput matters and losing a node (rebuilt on the next miss) is fine.

Redis deploys as a **replication group** — one primary + up to five replicas, Multi-AZ promoting a replica on failure; **Cluster Mode Disabled** = one shard (up to ~500 GB, single primary), **Enabled** = keyspace sharded across up to 500 shards for linear write scaling (client hits the **configuration endpoint**).

Zooming out — **the tier stacks, it doesn't replace**: the **source of truth** is always relational (RDS or Aurora); **availability** is Multi-AZ / Aurora replicas / Global Database; **connection pooling** is RDS Proxy whenever Lambda is involved; **read scaling** is read replicas first, then **ElastiCache** in front of the heavy repeated reads; and anything needing **sub-millisecond** latency (sessions, rate-limit counters, leaderboards) belongs in **ElastiCache**, not the database. Durable state in the DB; the hot subset in the cache.